# Experiment: CS-LSTM (convolutional social pooling)

Deo & Trivedi, CVPRW 2018. Each vehicle's history goes through a shared LSTM; neighbour encodings are placed on a 13 x 3 grid around the target (13 cells along the road, 3 lanes), convolved and pooled, then concatenated with the target's encoding and decoded by an LSTM. Paper (STRAP Table I) NGSIM RMSE: 0.57 / 1.26 / 2.11 / 3.18 / 4.50 m.

Part of the model comparison in `experiments/MODEL_ALTERNATIVES.md`. Run All; set `DEBUG = False` in section 2 for the real run.

## 0. Decisions & assumptions

Every choice that could change the result is listed here; the cell notes below repeat the ones that apply to each cell.

**Shared by every notebook (S)**

| # | Decision / assumption | Why | Departure from paper / alternative |
|---|---|---|---|
| S1 | Same data for every model: NGSIM us-101 + i-80 (DEBUG: first 250k rows of us-101), scene-level 70/10/20 split, seed 42, train-only normalization. | Differences in the results must come from the model, not the data. | Papers use their own splits, so published numbers are not like-for-like with ours. |
| S2 | Same training budget for every model: config.py defaults (12 epochs, Adam lr 5e-4 decayed x0.6 per epoch, batch 128, weight decay 1e-5, grad clip 1.0). | A fair comparison; per-model tuning would favour whichever model got tuned most. | Each paper's own schedule (e.g. CS-LSTM trains longer with lr 1e-3). Tune later with --mode tune if a model looks under-trained. |
| S3 | Loss = MSE + bivariate-Gaussian NLL + neighbour-goal MSE (`basic_loss`); the risk-scaled loss is used only where this notebook's M table says so (STRAP-R-based variants). Models without a goal head output zero goals, so their goal term is a constant. A model may add its own `aux_loss` (only the diffusion model does). | Keeps one training objective; the risk-scaled loss is STRAP-R's own contribution. | Loss values are therefore not comparable across models: compare RMSE only. |
| S4 | Best checkpoint = lowest validation loss (`fit` default); test RMSE is reported at 1-5 s for that checkpoint. | Same selection rule for every model. | Selecting on val RMSE would be closer to the reported metric. |
| S5 | KNOWN LIMITATION (not fixed): positions are relative to the target at each timestep, so the target's own history is always (0, 0) and lateral speed is always 0 (NGSIM gives longitudinal speed only). The target is seen through speed, acceleration and lane features; lateral context comes from neighbours. | Kept as is (user decision) so every model gets identical inputs. | Papers feed the target's own x/y track; lane-change prediction is handicapped for all models here. |
| S6 | Neighbours = up to 15 risk-selected vehicles (S/O-field > RISK_THRESHOLD), not every nearby vehicle. | Same scene construction as STRAP. | Grid/graph papers use all vehicles in range; ours see only the risk-selected ones. |
| S7 | Output = one bivariate Gaussian per future step (mu_x, mu_y, sigma_x, sigma_y, rho); RMSE uses the mean. Multimodal models are moment-matched to one Gaussian. | The `fit` / `evaluate_trajectory` contract and how STRAP's Table I is scored. | minADE/minFDE over modes would favour multimodal models. |

**Specific to this model (M)**

| # | Decision / assumption | Why | Departure from paper / alternative |
|---|---|---|---|
| M1 | Unimodal CS-LSTM (the 'CS-LSTM' row), not CS-LSTM(M) with 6 maneuver classes. | `fit` never shows the model the future, so maneuver labels cannot supervise a classifier. | CS-LSTM(M) needs lateral/longitudinal maneuver labels and teacher forcing in training. |
| M2 | Encoder input = all 11 features per vehicle through FC(11->32) + leaky ReLU + LSTM(64). | The paper feeds x/y tracks, but the target's track is all zeros here (S5). | Paper: FC(2->32) on x/y only. |
| M3 | Grid = 13 longitudinal cells of 4.57 m (15 ft, +-29.7 m) x 3 lanes; a neighbour's cell comes from its relative longitudinal position and relative lane (features 0 and 6, denormalized) at the last history step. | Paper geometry; lane comes from the lane feature instead of lateral metres. | Neighbours more than one lane away or outside +-29.7 m are dropped; two vehicles in one cell: the later one wins. |
| M4 | Only the risk-selected neighbours fill the grid (S6). | Same scenes as every model. | Paper fills every vehicle in range. |
| M5 | Social tensor: conv 3x3 (64) -> conv 3x1 (16) -> max-pool 2x1 -> 80 features; target dynamics FC(64->32); decoder LSTM(128) fed the same context at every step; leaky ReLU 0.1. | Sizes from the public CS-LSTM code. | - |
| M6 | sigma = softplus + 1e-3 (paper uses exp). | Same parameterisation as the other notebooks. | - |

## 1. Setup

**Decisions in this cell:** seed fixed to `SEED` (42) in every notebook so initialisation is not a source of differences (S1).

In [ ]:
import sys
from pathlib import Path

# Repo root = first folder upwards that contains src/ (works from experiments/ or the root).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.config import (
    BATCH_SIZE, BETA_LOSS, DEBUG_MAX_ROWS, DEVICE, DROPOUT, DT, LEARNING_RATE, LR_DECAY, N_V,
    NGSIM_LOCATIONS, NUM_EPOCHS, NORMALIZATION_FILE, SEED, STATE_DIM, TF_STEPS, TH_STEPS, WEIGHT_DECAY,
)
from src.data.loader import NGSIMDataLoader, create_dataloader
from src.data.normalization import load_stats, normalize_samples
from src.evaluation.evaluate import evaluate_trajectory
from src.models.strap import STRAP
from src.pipeline import collision_scores, metric_rows, test_groups
from src.train import fit, prepare_data
from experiments.benchmark import compare, save_result

torch.manual_seed(SEED)
print("device:", DEVICE)

## 2. Experiment config

**Decisions in this cell:** training budget is config.py's (S2); `use_risk_loss` as in the M table; collisions scored at 2 m like `--mode collisions`.

In [ ]:
EXP_NAME = "cs_lstm"          # unique per notebook -> experiments/results/<EXP_NAME>.json
DESCRIPTION = "CS-LSTM: LSTM encoders + 13x3 convolutional social pooling (unimodal)"

DEBUG = True                  # True: DEBUG_MAX_ROWS of us-101 for a quick check; False: full data
LOCATIONS = ["us-101"] if DEBUG else NGSIM_LOCATIONS

HPARAMS = dict(               # keyword args of src.train.fit
    num_epochs=NUM_EPOCHS,
    use_risk_loss=False,       # False = basic loss (STRAP-B style)
    beta=BETA_LOSS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    lr_decay=LR_DECAY,
    weight_decay=WEIGHT_DECAY,
)

RUN_COLLISIONS = True         # per-pair collision scoring on the test split (slow on full data)
COLLISION_THRESHOLD = 2.0     # metres, GT collision label distance

CKPT_DIR = ROOT / "experiments" / "checkpoints" / EXP_NAME

## 3. Data
Train-only normalization stats and the intention codebook are written to `CKPT_DIR`.

**Decisions in this cell:** same loader, split and train-only normalization as every other notebook (S1, S6); `norm_stats` is loaded here so the model (next section) can recover metres and lane numbers.

In [ ]:
loader = NGSIMDataLoader(locations=LOCATIONS, max_rows=DEBUG_MAX_ROWS if DEBUG else None)
train_s, val_s, centers = prepare_data(loader, save_dir=str(CKPT_DIR))
_, _, test_raw = loader.get_splits()   # cached; raw (unnormalized) test split
# Train-only mean/std; models that need metres or lane numbers undo the normalization with it.
norm_stats = load_stats(str(CKPT_DIR / NORMALIZATION_FILE))
print(f"train/val/test = {len(train_s)}/{len(val_s)}/{len(test_raw)}")

## 4. Model

Any `torch.nn.Module` works if `forward(states, mask, dims)` returns a 3-tuple:

| output | shape | meaning |
|---|---|---|
| `traj_dist` | `(B, T_f=50, 5)` | `[mu_x, mu_y, sigma_x, sigma_y, rho]` of the target, metres relative to its last observed position (x longitudinal, y lateral); `sigma > 0`, `rho` in (-1, 1) |
| `goals` | `(B, N-1, 4)` | neighbour goals; a model without a goal head returns `states.new_zeros(B, N - 1, 4)` |
| third | anything | ignored by training/eval (STRAP returns its risk field) |

Inputs: `states (B, T_h=30, N, 11)` normalized (vehicle 0 = target, rest neighbours, zero-padded),
`mask (B, N)` True = real vehicle, `dims (B, N, 2)` raw length/width in metres.
Feature index: 0 rel-longitudinal m, 1 rel-lateral m, 2 speed/30, 3 lateral speed (always 0), 4 accel/10,
5 type/2, 6 relative lane (target - vehicle, left = +), 7 length/10, 8 width/3, 9 S-field risk, 10 O-field risk.
Raw value = `states * std + mean` with `norm_stats`. STRAP's k-means codebook is only loaded if the model has a `risk_decoder`.

**Decisions in this cell:** unimodal (M1); 11-feature encoder (M2); grid from denormalized long. offset and relative lane at the last step (M3, M4); paper layer sizes (M5); softplus sigma (M6). Built after the data cell because it needs `norm_stats`.

In [ ]:
class CSLSTM(nn.Module):
    """Convolutional social pooling LSTM (Deo & Trivedi 2018), unimodal."""
    ROWS, COLS, CELL = 13, 3, 4.572   # M3: 13 x 3 grid, 15 ft cells

    def __init__(self, norm_stats, input_dim=STATE_DIM, emb=32, enc=64, dyn=32, dec=128, T_f=TF_STEPS):
        super().__init__()
        self.T_f = T_f
        self.register_buffer("mean", torch.tensor(norm_stats["mean"]))
        self.register_buffer("std", torch.tensor(norm_stats["std"]))
        self.embed = nn.Linear(input_dim, emb)                     # M2: all 11 features
        self.encoder = nn.LSTM(emb, enc, batch_first=True)
        self.dyn = nn.Linear(enc, dyn)
        self.conv1 = nn.Conv2d(enc, 64, 3)                         # (13, 3) -> (11, 1)
        self.conv2 = nn.Conv2d(64, 16, (3, 1))                     # (11, 1) -> (9, 1)
        self.pool = nn.MaxPool2d((2, 1), padding=(1, 0))           # (9, 1) -> (5, 1)
        self.decoder = nn.LSTM(dyn + 16 * 5, dec, batch_first=True)
        self.out = nn.Linear(dec, 5)

    def social_grid(self, states, mask, h):
        """Scatter neighbour encodings h (B, N, enc) into a (B, enc, 13, 3) grid."""
        B, N, E = h.shape
        raw = states[:, -1] * self.std + self.mean                 # last step, metres / lane numbers
        row = torch.floor(raw[..., 0] / self.CELL + self.ROWS / 2).long()
        col = (1 - torch.round(raw[..., 6])).long()                # lane +1 (left) -> 0, same -> 1, -1 (right) -> 2
        inside = mask & (row >= 0) & (row < self.ROWS) & (col >= 0) & (col < self.COLS)
        inside[:, 0] = False                                       # the target is not its own neighbour
        b, v = inside.nonzero(as_tuple=True)
        grid = h.new_zeros(B, self.ROWS * self.COLS, E)
        grid[b, row[b, v] * self.COLS + col[b, v]] = h[b, v]       # M3: same cell -> later vehicle wins
        return grid.view(B, self.ROWS, self.COLS, E).permute(0, 3, 1, 2)

    def forward(self, states, mask, dims=None):
        B, T_h, N, D_in = states.shape
        x = F.leaky_relu(self.embed(states.transpose(1, 2).reshape(B * N, T_h, D_in)), 0.1)
        _, (h, _) = self.encoder(x)
        h = h[-1].view(B, N, -1)                                   # one encoding per vehicle
        grid = self.social_grid(states, mask, h)
        social = self.pool(F.leaky_relu(self.conv2(F.leaky_relu(self.conv1(grid), 0.1)), 0.1)).flatten(1)
        context = torch.cat([F.leaky_relu(self.dyn(h[:, 0]), 0.1), social], dim=-1)
        out, _ = self.decoder(context[:, None].expand(-1, self.T_f, -1))
        out = self.out(out)
        traj = torch.cat([out[..., :2], F.softplus(out[..., 2:4]) + 1e-3, torch.tanh(out[..., 4:])], dim=-1)  # M6
        return traj, states.new_zeros(B, N - 1, 4), None


def build_model():
    return CSLSTM(norm_stats)

model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {n_params:,} parameters")

## 5. Train

**Decisions in this cell:** `fit` with the config budget (S2); best checkpoint by val loss (S4). Loss curves are only comparable within this notebook (S3).

In [ ]:
t0 = time.time()
training_log = fit(model, train_s, val_s, centers, save_dir=str(CKPT_DIR), **HPARAMS)
train_minutes = (time.time() - t0) / 60

log_df = pd.DataFrame(training_log)
display(log_df)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
log_df.plot(x="epoch", y=["train_loss", "val_loss"], marker=".", ax=axes[0], title="Loss")
log_df.plot(x="epoch", y="val_rmse", marker=".", ax=axes[1], title="Val RMSE (m)")
plt.tight_layout(); plt.show()

## 6. Evaluate (best checkpoint) — RMSE 1s…5s overall, per site, zero-neighbour scenes

**Decisions in this cell:** RMSE of the predicted mean at 1-5 s on the test split, overall / per site / target-only scenes (S7).

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / "best_model.pt", map_location=DEVICE, weights_only=True))
model.eval()
test_s = normalize_samples(test_raw, norm_stats)

test_metrics = {}
for label, samples in test_groups(test_s).items():
    dl = create_dataloader(samples, batch_size=32, shuffle=False)
    test_metrics[label] = {**evaluate_trajectory(model, dl, DEVICE), "n": len(samples)}
display(pd.DataFrame(test_metrics).T)

## 7. Collision prediction (optional)
Model's predicted target vs constant-velocity neighbours, every risk metric at its default threshold.

**Decisions in this cell:** neighbours are rolled forward at constant velocity and candidates are the nearest vehicles at the last history step, as in `--mode collisions`; only the target trajectory comes from this model.

In [ ]:
collisions = None
if RUN_COLLISIONS:
    scores, gt, _ = collision_scores(model, loader, test_raw, norm_stats, COLLISION_THRESHOLD)
    collisions = {"threshold": COLLISION_THRESHOLD, "n_pairs": len(gt),
                  "n_gt_collisions": int(gt.sum()), "rows": metric_rows(scores, gt)}
    print(f"{len(gt)} pairs, {int(gt.sum())} GT collisions")
    display(pd.DataFrame(collisions["rows"]).set_index("metric"))

## 8. Save result

**Decisions in this cell:** one JSON per `EXP_NAME`; a rerun overwrites it.

In [ ]:
save_result({
    "name": EXP_NAME,
    "description": DESCRIPTION,
    "model_class": type(model).__name__,
    "n_params": n_params,
    "hparams": HPARAMS,
    "data": {"locations": LOCATIONS, "debug": DEBUG},
    "train_minutes": train_minutes,
    "training_log": training_log,
    "test": test_metrics,
    "collisions": collisions,
})

## 9. Comparison & benchmarking — all results in `experiments/results/`

Ranked by overall test `rmse_avg` (best per column highlighted). Check the `data` column:
debug and full-data runs are listed together but are not comparable.
Also written to `experiments/comparison_rmse.csv` / `comparison_collisions.csv`.

**Decisions in this cell:** compares every saved result; only rows with the same `data` value are like-for-like.

In [ ]:
rmse_df, collision_df = compare()